# YOLO Tracking Benchmark Notebook (Webcam or Video)

**Goal:** Compare performance across YOLO model sizes on your machine (CPU-only friendly), for either:
- **Live webcam tracking** (`source=0`), or
- **Local video file** (`source="path/to/video.mp4"`)

---

## Mind map (how this notebook is organized)

1) **Setup**
   - Install + imports
   - Hardware / environment snapshot

2) **Experiment config**
   - Choose *source* (webcam vs video)
   - Choose *task* (track vs predict)
   - Choose model variants (n/s/m/l/x)
   - Choose runtime window (seconds) + sampling interval

3) **Instrumentation**
   - Frame timing → FPS + latency stats
   - Process + system metrics (Task Manager-style, but loggable)
     - CPU %, RAM, CPU frequency, threads, I/O

4) **Runs**
   - Warmup (stabilize caches / JIT)
   - Repeated trials per model

5) **Outputs**
   - `results_summary.csv` (one row per model/trial)
   - `results_timeseries.csv` (many rows per sample, per run)
   - Plots + quick interpretation prompts

---

> **State machine (mental model)**
>
> **CONFIG → WARMUP → MEASURE → SUMMARIZE → COMPARE → SAVE**


In [ ]:
# =========================
# SETUP (STATE: CONFIG)
# =========================
# Mind map note:
# - Keep installs minimal & reproducible.
# - CPU-only: ultralytics + psutil is enough.
# - Optional: OpenCV for preview windows.

!pip -q install ultralytics psutil pandas tqdm matplotlib opencv-python


In [ ]:
# =========================
# IMPORTS (STATE: CONFIG)
# =========================

import os
import time
import json
import math
import platform
import statistics
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple

import pandas as pd
import psutil
from tqdm import tqdm

import matplotlib.pyplot as plt

# Optional (for local preview windows). If unavailable, benchmarking still works.
try:
    import cv2
    _HAS_CV2 = True
except Exception:
    _HAS_CV2 = False

from ultralytics import YOLO


In [ ]:
# =========================
# EXPERIMENT CONFIG (STATE: CONFIG)
# =========================
# Mind map note:
# - Keep *all knobs in one place* so classmates can change 2-3 lines and run.
# - Prefer time-based runs for webcam repeatability.

# Choose source mode:
#   "webcam"  -> uses webcam device 0
#   "video"   -> uses VIDEO_PATH
RUN_MODE = "webcam"   # "webcam" or "video"
VIDEO_PATH = "video.mp4"  # used only if RUN_MODE == "video"

# Choose task:
#   "track"   -> detection + tracking IDs
#   "predict" -> detection only (sometimes useful as a baseline)
TASK = "track"        # "track" or "predict"

# Model variants to compare (Ultralytics will auto-download weights if missing)
# Common: yolov8n/s/m/l/x.pt
MODELS = [
    "yolov8n.pt",
    "yolov8s.pt",
    "yolov8m.pt",
    # "yolov8l.pt",
    # "yolov8x.pt",
]

# Runtime control
RUN_SECONDS = 20          # measurement window per trial
WARMUP_SECONDS = 5        # warmup window per trial (excluded from metrics)
TRIALS_PER_MODEL = 2      # repeated runs → compute mean/std later

# Detector params
IMGSZ = 640
CONF = 0.25
IOU = 0.5

# Instrumentation
METRIC_SAMPLE_PERIOD_S = 0.5   # system/proc sample interval
FRAME_TIME_KEEP_EVERY = 1      # keep timing for every Nth frame result

# Output
OUTPUT_DIR = "outputs_yolo_benchmark"
SAVE_CSV = True

# Webcam preview (optional):
# - In many notebook environments, cv2.imshow may not work.
# - If it works on your machine, this is a nice live demo mode.
PREVIEW_LIVE = False
PREVIEW_WINDOW_NAME = "YOLO Preview (press q to quit)"

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# =========================
# ENVIRONMENT SNAPSHOT (STATE: CONFIG)
# =========================
# Mind map note:
# - Logging environment info makes your report more credible and reproducible.

def environment_snapshot() -> Dict[str, str]:
    info = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "processor": platform.processor(),
        "machine": platform.machine(),
        "cpu_count_logical": str(psutil.cpu_count(logical=True)),
        "cpu_count_physical": str(psutil.cpu_count(logical=False)),
        "ram_total_gb": f"{psutil.virtual_memory().total / (1024**3):.2f}",
    }
    # Optional CPU freq
    freq = psutil.cpu_freq()
    if freq:
        info["cpu_freq_mhz_current"] = f"{freq.current:.0f}"
        info["cpu_freq_mhz_max"] = f"{freq.max:.0f}"
    return info

env = environment_snapshot()
pd.DataFrame([env])


In [ ]:
# =========================
# INSTRUMENTATION (STATE: CONFIG → MEASURE)
# =========================
# Mind map note:
# - We log two streams:
#   A) frame-level timing (FPS / latency)
#   B) periodic system/process metrics (Task Manager-like)
#
# - For fair comparisons: same scene, same duration, multiple trials.

PROC = psutil.Process()  # current Python process

def _prime_cpu_percent():
    # These need a first call to become meaningful.
    PROC.cpu_percent(interval=None)
    psutil.cpu_percent(interval=None)

def sample_metrics() -> Dict[str, Optional[float]]:
    """Task Manager-style metrics, but per-process + system."""
    m = psutil.virtual_memory()
    io = PROC.io_counters() if hasattr(PROC, "io_counters") else None
    freq = psutil.cpu_freq()

    return {
        "ts": time.time(),

        # Process-scoped
        "proc_cpu_pct": PROC.cpu_percent(interval=None),
        "proc_rss_gb": PROC.memory_info().rss / (1024**3),
        "proc_vms_gb": PROC.memory_info().vms / (1024**3),
        "proc_threads": float(PROC.num_threads()),

        "proc_read_mb": (io.read_bytes / (1024**2)) if io else None,
        "proc_write_mb": (io.write_bytes / (1024**2)) if io else None,

        # System-scoped
        "sys_cpu_pct": psutil.cpu_percent(interval=None),
        "sys_ram_pct": m.percent,
        "sys_ram_used_gb": m.used / (1024**3),

        # CPU frequency proxy (boost/throttle)
        "cpu_freq_mhz": (freq.current if freq else None),
    }

def summarize_frame_times(frame_times_s: List[float]) -> Dict[str, float]:
    """Compute robust stats from per-frame durations."""
    if not frame_times_s:
        return {"fps_mean": float("nan"), "frame_ms_mean": float("nan")}
    # convert to ms for readability
    ms = [t * 1000.0 for t in frame_times_s]
    ms_sorted = sorted(ms)
    def pct(p):
        if not ms_sorted:
            return float("nan")
        k = (len(ms_sorted)-1) * (p/100.0)
        f = math.floor(k); c = math.ceil(k)
        if f == c:
            return ms_sorted[int(k)]
        return ms_sorted[f] + (ms_sorted[c] - ms_sorted[f]) * (k - f)

    mean_ms = statistics.mean(ms)
    p50 = pct(50); p90 = pct(90); p95 = pct(95); p99 = pct(99)
    fps = 1000.0 / mean_ms if mean_ms > 0 else float("inf")
    return {
        "fps_mean": fps,
        "frame_ms_mean": mean_ms,
        "frame_ms_p50": p50,
        "frame_ms_p90": p90,
        "frame_ms_p95": p95,
        "frame_ms_p99": p99,
    }

def summarize_metric_series(samples: List[Dict[str, Optional[float]]]) -> Dict[str, Optional[float]]:
    """Aggregate periodic metrics into averages/peaks."""
    if not samples:
        return {}

    df = pd.DataFrame(samples)
    out = {}
    for col in ["proc_cpu_pct","proc_rss_gb","proc_threads","sys_cpu_pct","sys_ram_pct","cpu_freq_mhz"]:
        if col in df.columns:
            series = pd.to_numeric(df[col], errors="coerce").dropna()
            if len(series) == 0:
                out[f"{col}_avg"] = None
                out[f"{col}_max"] = None
            else:
                out[f"{col}_avg"] = float(series.mean())
                out[f"{col}_max"] = float(series.max())
    return out


In [ ]:
# =========================
# RUNNER (STATE: WARMUP → MEASURE → SUMMARIZE)
# =========================
# Mind map note:
# - We run each model with:
#     warmup seconds  (ignored)
#     measure seconds (logged)
# - We stream results so we can time them.
# - For webcam: stop after RUN_SECONDS (not frame count).
# - For video: also time-based to keep parity.

@dataclass
class RunConfig:
    run_mode: str
    video_path: str
    task: str
    imgsz: int
    conf: float
    iou: float
    run_seconds: int
    warmup_seconds: int
    metric_sample_period_s: float
    frame_time_keep_every: int
    preview_live: bool

def _get_source(cfg: RunConfig):
    if cfg.run_mode == "webcam":
        return 0
    return cfg.video_path

def run_once(model_name: str, cfg: RunConfig, trial_idx: int) -> Tuple[Dict, pd.DataFrame]:
    """Returns (summary_row, timeseries_df)."""
    _prime_cpu_percent()

    model = YOLO(model_name)
    source = _get_source(cfg)

    # Pick method and create stream iterator
    if cfg.task == "track":
        iterator = model.track(
            source=source,
            stream=True,
            persist=True,
            imgsz=cfg.imgsz,
            conf=cfg.conf,
            iou=cfg.iou,
            verbose=False
        )
    elif cfg.task == "predict":
        iterator = model.predict(
            source=source,
            stream=True,
            imgsz=cfg.imgsz,
            conf=cfg.conf,
            iou=cfg.iou,
            verbose=False
        )
    else:
        raise ValueError("TASK must be 'track' or 'predict'")

    # Timing + periodic metrics
    frame_times = []
    metric_samples = []
    timeseries_rows = []

    start = time.time()
    warmup_end = start + cfg.warmup_seconds
    end = warmup_end + cfg.run_seconds

    next_metric_t = start

    # Preview state (optional)
    use_preview = cfg.preview_live and _HAS_CV2 and (cfg.run_mode == "webcam")
    if cfg.preview_live and not _HAS_CV2:
        print("PREVIEW_LIVE=True but cv2 isn't available. Continuing without preview.")

    last_frame_t = time.perf_counter()
    kept = 0
    for r in iterator:
        now = time.time()
        if now >= end:
            break

        # Sample metrics at a fixed cadence (not every frame)
        if now >= next_metric_t:
            sm = sample_metrics()
            sm.update({
                "model": model_name,
                "trial": trial_idx,
                "phase": "warmup" if now < warmup_end else "measure",
            })
            metric_samples.append(sm)
            timeseries_rows.append(sm)
            next_metric_t = now + cfg.metric_sample_period_s

        # Frame timing (exclude warmup)
        if now >= warmup_end:
            if kept % cfg.frame_time_keep_every == 0:
                t = time.perf_counter()
                frame_times.append(t - last_frame_t)
                last_frame_t = t
            kept += 1

        # Optional live preview (webcam only)
        if use_preview:
            # Ultralytics provides an annotated image in r.plot()
            frame = r.plot()
            cv2.imshow(PREVIEW_WINDOW_NAME, frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    if use_preview:
        cv2.destroyAllWindows()

    # Summaries
    frame_summary = summarize_frame_times(frame_times)
    metric_summary = summarize_metric_series([s for s in metric_samples if s.get("phase") == "measure"])

    summary = {
        "model": model_name,
        "trial": trial_idx,
        "run_mode": cfg.run_mode,
        "source": "webcam(0)" if cfg.run_mode == "webcam" else cfg.video_path,
        "task": cfg.task,
        "imgsz": cfg.imgsz,
        "conf": cfg.conf,
        "iou": cfg.iou,
        "warmup_seconds": cfg.warmup_seconds,
        "run_seconds": cfg.run_seconds,
        "frames_timed": len(frame_times),
        **frame_summary,
        **metric_summary,
    }

    ts_df = pd.DataFrame(timeseries_rows)
    return summary, ts_df

cfg = RunConfig(
    run_mode=RUN_MODE,
    video_path=VIDEO_PATH,
    task=TASK,
    imgsz=IMGSZ,
    conf=CONF,
    iou=IOU,
    run_seconds=RUN_SECONDS,
    warmup_seconds=WARMUP_SECONDS,
    metric_sample_period_s=METRIC_SAMPLE_PERIOD_S,
    frame_time_keep_every=FRAME_TIME_KEEP_EVERY,
    preview_live=PREVIEW_LIVE
)
cfg


In [ ]:
# =========================
# EXPERIMENT LOOP (STATE: MEASURE → SUMMARIZE)
# =========================
# Mind map note:
# - Multiple trials reduce noise.
# - Keep the webcam scene consistent across runs.

all_summaries = []
all_timeseries = []

for model_name in MODELS:
    for trial in range(1, TRIALS_PER_MODEL + 1):
        print(f"Running {model_name} (trial {trial}/{TRIALS_PER_MODEL})...")
        summary, ts = run_once(model_name=model_name, cfg=cfg, trial_idx=trial)
        all_summaries.append(summary)
        all_timeseries.append(ts)

df_summary = pd.DataFrame(all_summaries)
df_timeseries = pd.concat(all_timeseries, ignore_index=True) if all_timeseries else pd.DataFrame()

df_summary


In [ ]:
# =========================
# AGGREGATION (STATE: COMPARE)
# =========================
# Mind map note:
# - Report both per-trial and aggregated stats (mean/std).
# - Sort by FPS to make the story obvious.

if len(df_summary) == 0:
    raise RuntimeError("No results. Check RUN_MODE/VIDEO_PATH and webcam permissions.")

# Per-model aggregation across trials
agg = (df_summary
       .groupby("model", as_index=False)
       .agg(
           fps_mean_mean=("fps_mean", "mean"),
           fps_mean_std=("fps_mean", "std"),
           frame_ms_mean_mean=("frame_ms_mean", "mean"),
           proc_cpu_pct_avg_mean=("proc_cpu_pct_avg", "mean"),
           proc_rss_gb_avg_mean=("proc_rss_gb_avg", "mean"),
           cpu_freq_mhz_avg_mean=("cpu_freq_mhz_avg", "mean"),
       )
      )

agg = agg.sort_values("fps_mean_mean", ascending=False)
agg


In [ ]:
# =========================
# SAVE OUTPUTS (STATE: SAVE)
# =========================
# Mind map note:
# - Two CSVs:
#   1) summary: one row per run
#   2) timeseries: one row per metric sample
#
# These are the artifacts you include in your lab report.

summary_csv = os.path.join(OUTPUT_DIR, "results_summary.csv")
timeseries_csv = os.path.join(OUTPUT_DIR, "results_timeseries.csv")
env_json = os.path.join(OUTPUT_DIR, "environment_snapshot.json")

if SAVE_CSV:
    df_summary.to_csv(summary_csv, index=False)
    df_timeseries.to_csv(timeseries_csv, index=False)
    with open(env_json, "w", encoding="utf-8") as f:
        json.dump(env, f, indent=2)
    print("Saved:", summary_csv)
    print("Saved:", timeseries_csv)
    print("Saved:", env_json)


In [ ]:
# =========================
# PLOTS (STATE: COMPARE)
# =========================
# Mind map note:
# - Minimal plots that tell the story fast:
#   A) FPS vs model size
#   B) CPU% vs model size
#   C) RAM vs model size
# - For timeseries, overlay CPU% over time for one model/trial if desired.

# A) FPS by model (aggregated)
plt.figure()
plt.bar(agg["model"], agg["fps_mean_mean"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("FPS (mean over trials)")
plt.title("YOLO Performance (FPS) by Model")
plt.tight_layout()
plt.show()

# B) Process CPU% (aggregated)
plt.figure()
plt.bar(agg["model"], agg["proc_cpu_pct_avg_mean"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Process CPU% (mean over trials)")
plt.title("CPU Load by Model")
plt.tight_layout()
plt.show()

# C) Process RAM (aggregated)
plt.figure()
plt.bar(agg["model"], agg["proc_rss_gb_avg_mean"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Process RSS (GB, mean over trials)")
plt.title("Memory Usage by Model")
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# OPTIONAL: TIMESERIES VIEW (STATE: COMPARE)
# =========================
# Mind map note:
# - Timeseries is great for showing thermal throttling / steady-state behavior.
# - Pick one model + trial and visualize CPU% over time.

if len(df_timeseries) > 0:
    # pick first model/trial measured phase
    ex = df_timeseries[df_timeseries["phase"] == "measure"].copy()
    if len(ex) > 0:
        m0 = ex["model"].iloc[0]
        t0 = ex["trial"].iloc[0]
        ex = ex[(ex["model"] == m0) & (ex["trial"] == t0)].copy()
        ex["t_rel_s"] = ex["ts"] - ex["ts"].min()

        plt.figure()
        plt.plot(ex["t_rel_s"], ex["proc_cpu_pct"])
        plt.xlabel("Time (s)")
        plt.ylabel("Process CPU%")
        plt.title(f"CPU% over time (example: {m0}, trial {t0})")
        plt.tight_layout()
        plt.show()



## Semantic accuracy study (human-in-the-loop)

### Why this exists (mind-map)

**Goal:** test whether **semantic correctness** (did the model name the object correctly?) correlates with **model size**.

**Idea:** while running webcam/video inference we **sample a few detections**, save the **cropped boxes**, then **quiz a human** afterwards:
- model says: *toothbrush*
- you say: **no, it's a pen**
- we log that correction

This creates a lightweight **human-labeled dataset** on-the-fly, per model size, without needing a full benchmark dataset.

### State machine (high level)

1. **RUN** (webcam/video) -> collect YOLO detections
2. **SAMPLE** some boxes -> save crops + metadata
3. **REVIEW** crops -> human confirms/corrects labels
4. **SCORE** per model -> accuracy + confusion-ish summaries
5. **COMPARE** accuracy vs FPS / latency across model sizes


### Install / imports (only needed for this section)

In [ ]:
# If you don't have widgets already, uncomment:
# !pip -q install ipywidgets

import os, time
import pandas as pd

from IPython.display import display, clear_output
try:
    import ipywidgets as widgets
    _HAS_WIDGETS = True
except Exception:
    _HAS_WIDGETS = False

import cv2

### Capture settings

In [ ]:
# === Capture configuration ===
CAPTURE_QUIZ_SAMPLES = True      # turn on/off sampling crops during a run
CAPTURE_DIR = "captures"         # folder where crops + metadata are written
CAPTURE_EVERY_SECONDS = 1.0      # sample at most once per X seconds
MAX_CAPTURES_PER_RUN = 120       # cap so you don't create thousands of crops
MIN_CONF_FOR_CAPTURE = 0.35      # ignore very low-confidence boxes

os.makedirs(CAPTURE_DIR, exist_ok=True)

### Helper: save crops from a YOLO result

In [ ]:
def _xyxy_to_int(xyxy, w, h):
    x1, y1, x2, y2 = [int(round(float(v))) for v in xyxy]
    x1 = max(0, min(x1, w - 1))
    x2 = max(0, min(x2, w - 1))
    y1 = max(0, min(y1, h - 1))
    y2 = max(0, min(y2, h - 1))
    if x2 <= x1: x2 = min(w - 1, x1 + 1)
    if y2 <= y1: y2 = min(h - 1, y1 + 1)
    return x1, y1, x2, y2

def capture_crops_from_result(result, model_tag, run_tag, capture_dir=CAPTURE_DIR, min_conf=MIN_CONF_FOR_CAPTURE):
    \"\"\"
    Saves cropped detections from an Ultralytics result to disk + returns rows of metadata.

    Mind-map:
    - We want *semantic evaluation*, not perfect tracking evaluation.
    - So we sample a few boxes, save crops, and ask a human later.
    \"\"\"
    rows = []

    if result is None or result.boxes is None or len(result.boxes) == 0:
        return rows

    img = result.orig_img  # numpy BGR image
    if img is None:
        return rows

    h, w = img.shape[:2]

    boxes = result.boxes
    xyxy = boxes.xyxy.cpu().numpy()
    conf = boxes.conf.cpu().numpy()
    cls = boxes.cls.cpu().numpy().astype(int)

    # Track IDs may not exist depending on tracker/back-end
    track_ids = None
    if hasattr(boxes, "id") and boxes.id is not None:
        try:
            track_ids = boxes.id.cpu().numpy().astype(int)
        except Exception:
            track_ids = None

    names = result.names if hasattr(result, "names") else None

    ts = time.time()
    for i in range(len(xyxy)):
        if float(conf[i]) < float(min_conf):
            continue

        x1, y1, x2, y2 = _xyxy_to_int(xyxy[i], w, h)
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        pred_cls = int(cls[i])
        pred_name = names.get(pred_cls, str(pred_cls)) if isinstance(names, dict) else str(pred_cls)
        tid = int(track_ids[i]) if track_ids is not None else None

        fname = f"{model_tag}__{run_tag}__{int(ts*1000)}__i{i}__{pred_name}.jpg"
        fpath = os.path.join(capture_dir, fname)

        cv2.imwrite(fpath, crop)

        rows.append({
            "model": model_tag,
            "run": run_tag,
            "t": ts,
            "file": fpath,
            "pred_name": pred_name,
            "pred_cls": pred_cls,
            "conf": float(conf[i]),
            "track_id": tid,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "frame_w": w, "frame_h": h,
        })

    return rows

### Hook: integrate crop capture into the benchmark runner

If you already ran benchmarks above, you can re-run with capture enabled. **This wrapper** opportunistically saves crops during the MEASURE window.

In [ ]:
def run_benchmark_with_capture(
    model_path: str,
    source,
    task: str = "track",
    imgsz: int = 640,
    conf: float = 0.25,
    iou: float = 0.5,
    warmup_seconds: float = 3.0,
    bench_seconds: float = 15.0,
    run_tag: str = "run",
):
    \"\"\"
    State machine:
      WARMUP -> MEASURE
      During MEASURE: every CAPTURE_EVERY_SECONDS we SAMPLE a few boxes and save crops.
    \"\"\"
    model = YOLO(model_path)

    if task == "track":
        iterator = model.track(source=source, stream=True, persist=True, imgsz=imgsz, conf=conf, iou=iou, verbose=False)
    else:
        iterator = model.predict(source=source, stream=True, imgsz=imgsz, conf=conf, iou=iou, verbose=False)

    frame_times = []

    proc = psutil.Process()
    proc.cpu_percent(interval=None)
    psutil.cpu_percent(interval=None)

    sys_samples = []
    capture_rows = []
    last_cap_t = -1e9

    t_start = time.perf_counter()
    t_last = t_start

    total_target = warmup_seconds + bench_seconds

    for r in iterator:
        now = time.perf_counter()
        dt = now - t_last
        t_last = now

        elapsed = now - t_start

        # sample sys/proc every ~0.5s
        if len(sys_samples) == 0 or (sys_samples[-1]["t_perf"] + 0.5) <= now:
            sys_samples.append({
                "t_perf": now,
                "model": os.path.basename(model_path),
                "run": run_tag,
                "proc_cpu_pct": proc.cpu_percent(interval=None),
                "proc_rss_gb": proc.memory_info().rss / (1024**3),
                "sys_cpu_pct": psutil.cpu_percent(interval=None),
                "sys_ram_pct": psutil.virtual_memory().percent,
                "cpu_freq_mhz": (psutil.cpu_freq().current if psutil.cpu_freq() else None),
            })

        if elapsed >= warmup_seconds:
            frame_times.append(dt)

            if CAPTURE_QUIZ_SAMPLES:
                measure_elapsed = elapsed - warmup_seconds
                if (measure_elapsed - last_cap_t) >= CAPTURE_EVERY_SECONDS and len(capture_rows) < MAX_CAPTURES_PER_RUN:
                    last_cap_t = measure_elapsed
                    capture_rows.extend(
                        capture_crops_from_result(
                            result=r,
                            model_tag=os.path.basename(model_path).replace(".pt",""),
                            run_tag=run_tag,
                        )
                    )

        if elapsed >= total_target:
            break

    if len(frame_times) == 0:
        raise RuntimeError("No frames recorded in MEASURE window. Check webcam/video source.")

    avg_dt = sum(frame_times)/len(frame_times)
    fps = 1.0/avg_dt

    summary = {
        "model": os.path.basename(model_path),
        "task": task,
        "imgsz": imgsz,
        "conf": conf,
        "iou": iou,
        "warmup_seconds": warmup_seconds,
        "bench_seconds": bench_seconds,
        "frames_measured": len(frame_times),
        "fps": fps,
        "avg_frame_ms": avg_dt*1000.0,
        "captures": len(capture_rows),
    }

    return summary, pd.DataFrame(sys_samples), pd.DataFrame(capture_rows)

### Run capture experiments across model sizes

In [ ]:
CAPTURE_RESULTS = []
CAPTURE_SYS_TS = []
CAPTURE_CROPS = []

run_id = int(time.time())

for m in models:
    tag = f"{run_id}_{m.replace('.pt','')}"
    print("Capturing:", m)
    summ, sys_ts, crops = run_benchmark_with_capture(
        model_path=m,
        source=SOURCE,
        task="track",
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        warmup_seconds=WARMUP_SECONDS,
        bench_seconds=BENCH_SECONDS,
        run_tag=tag,
    )
    CAPTURE_RESULTS.append(summ)
    CAPTURE_SYS_TS.append(sys_ts)
    CAPTURE_CROPS.append(crops)

df_capture_summary = pd.DataFrame(CAPTURE_RESULTS)
df_capture_sys = pd.concat(CAPTURE_SYS_TS, ignore_index=True) if CAPTURE_SYS_TS else pd.DataFrame()
df_capture_crops = pd.concat(CAPTURE_CROPS, ignore_index=True) if CAPTURE_CROPS else pd.DataFrame()

display(df_capture_summary)
print("Crops:", len(df_capture_crops), "Sys samples:", len(df_capture_sys))

### Human review UI (quiz)

In [ ]:
def review_crops_interactively(df_crops: pd.DataFrame, out_csv: str = "human_labels.csv", shuffle: bool = True):
    \"\"\"
    Human-in-the-loop labeling UI.

    Mind-map:
    - show crop
    - show model prediction
    - ask user: correct? if not, what is it?
    - log correction
    \"\"\"
    if df_crops is None or len(df_crops) == 0:
        raise ValueError("No crops to review. Run capture first.")

    df = df_crops.copy().reset_index(drop=True)
    if shuffle:
        df = df.sample(frac=1.0, random_state=0).reset_index(drop=True)

    for col in ["is_correct", "true_label", "notes"]:
        if col not in df.columns:
            df[col] = None

    if not _HAS_WIDGETS:
        print("ipywidgets not available. Saving template CSV for manual annotation:", out_csv)
        df.to_csv(out_csv, index=False)
        return df

    idx = 0

    img_widget = widgets.Image(format='jpg')
    title = widgets.HTML()
    pred = widgets.HTML()

    correct_btn = widgets.Button(description="✅ Correct", button_style="success")
    wrong_btn = widgets.Button(description="❌ Wrong", button_style="danger")
    skip_btn = widgets.Button(description="⏭ Skip", button_style="warning")

    true_text = widgets.Text(description="True:", placeholder="e.g., pen")
    notes_text = widgets.Text(description="Notes:", placeholder="optional")

    progress = widgets.IntProgress(value=0, min=0, max=len(df))
    out = widgets.Output()

    def load_row(i):
        nonlocal idx
        idx = i
        row = df.loc[idx]
        with open(row["file"], "rb") as f:
            img_widget.value = f.read()

        title.value = f"<b>Sample {idx+1}/{len(df)}</b> — model: <code>{row['model']}</code> run: <code>{row['run']}</code>"
        pred.value = f"Prediction: <b>{row['pred_name']}</b> (conf={row['conf']:.2f})"
        true_text.value = "" if pd.isna(row["true_label"]) else str(row["true_label"])
        notes_text.value = "" if pd.isna(row["notes"]) else str(row["notes"])
        progress.value = idx + 1

    def save_and_advance():
        df.to_csv(out_csv, index=False)
        nxt = idx + 1
        if nxt >= len(df):
            with out:
                clear_output()
                print("Done! Saved:", out_csv)
            return
        load_row(nxt)

    def on_correct(_):
        df.at[idx, "is_correct"] = True
        df.at[idx, "true_label"] = df.at[idx, "pred_name"] if not true_text.value else true_text.value
        df.at[idx, "notes"] = notes_text.value
        save_and_advance()

    def on_wrong(_):
        df.at[idx, "is_correct"] = False
        df.at[idx, "true_label"] = true_text.value if true_text.value else None
        df.at[idx, "notes"] = notes_text.value
        save_and_advance()

    def on_skip(_):
        df.at[idx, "is_correct"] = None
        df.at[idx, "true_label"] = true_text.value if true_text.value else None
        df.at[idx, "notes"] = notes_text.value
        save_and_advance()

    correct_btn.on_click(on_correct)
    wrong_btn.on_click(on_wrong)
    skip_btn.on_click(on_skip)

    ui = widgets.VBox([
        title,
        img_widget,
        pred,
        widgets.HBox([correct_btn, wrong_btn, skip_btn]),
        true_text,
        notes_text,
        progress,
        out
    ])

    load_row(0)
    display(ui)
    return df

### Score semantic accuracy and compare to speed

In [ ]:
def summarize_semantic_accuracy(df_annot: pd.DataFrame):
    df = df_annot.copy()
    df = df[df["is_correct"].notna()].copy()
    if len(df) == 0:
        raise ValueError("No labeled rows (is_correct).")

    acc = df.groupby("model")["is_correct"].mean().reset_index(name="semantic_acc")
    n = df.groupby("model")["is_correct"].size().reset_index(name="n_labeled")
    out = acc.merge(n, on="model")
    return out.sort_values("semantic_acc", ascending=False)

# Workflow:
# annotated = review_crops_interactively(df_capture_crops, out_csv="human_labels.csv")
# semantic = summarize_semantic_accuracy(annotated)
# merged = semantic.merge(df_capture_summary[["model","fps","avg_frame_ms"]], on="model", how="left")
# display(merged)

# Correlation plot (optional)
# import matplotlib.pyplot as plt
# plt.figure()
# plt.scatter(merged["fps"], merged["semantic_acc"])
# plt.xlabel("FPS")
# plt.ylabel("Semantic accuracy (human-labeled)")
# plt.title("Semantic accuracy vs FPS")
# plt.show()

---

## Notes for your lab write-up

- **Fairness:** Keep the webcam view consistent across runs (lighting + scene).
- **Noise control:** Run ≥2 trials/model; report mean ± std.
- **Interpretation prompts:**
  - How much FPS do you lose as you scale from `n → s → m → l`?
  - Does CPU% saturate at 100%? If yes, you’re CPU-bound.
  - Does CPU frequency drop over time (throttling)? Use the timeseries plot.
  - Does memory grow materially with model size?

If you want tracking-quality metrics (ID switches, IDF1, MOTA), you’ll need a labeled dataset (e.g., MOT-style) — happy to extend this notebook for that if your lab requires it.
